# Secure functions

A `~function` has its strings and constants obfuscated in the compiled
chunk, so `strings` on a dump finds nothing.

**This is obfuscation, not encryption.** Read that sentence before the
rest of the notebook — the last section is the Lab reading straight
through it.

## The syntax is a statement, not an expression

`~function name(...)` or `local ~function name(...)`. It is *not*
`local f = ~function() end` — `~` is bitwise NOT in an expression, so
that parses as "complement of a function value" and fails.

In [ ]:
local ~function hidden(s)
  return "secret:" .. s
end

print(hidden("x"))

In [ ]:
-- the expression form is a different thing entirely, and an error
print(pcall(load, "local f = ~function() end"))
print(select(2, pcall(function() return ~print end)))

## What it hides

Compile a function that contains a distinctive literal, dump it, and
look for the literal in the bytes.

In [ ]:
local ~function vault()
  local phrase = "MARKER-IN-A-SECURE-FUNCTION"
  return phrase
end

local dump = string.dump(vault)
print($"dump is {#dump} bytes")
print("literal found in the clear?", dump:find("MARKER-IN-A-SECURE-FUNCTION", 1, true) ~= nil)
print("and it still runs:", vault())

An ordinary function, for contrast.

In [ ]:
local function plain()
  local phrase = "MARKER-IN-A-PLAIN-FUNCTION"
  return phrase
end

print("literal found in the clear?",
  string.dump(plain):find("MARKER-IN-A-PLAIN-FUNCTION", 1, true) ~= nil)

## A shared literal is hidden too, and that took a security release

Lua 5.5 stores each distinct string in a dump **once** and refers back
to it by index. So a literal used by both a secure function and
ordinary code has exactly one stored copy — at whichever site was
written first, which is not always the secure one.

In `5.5.1_build1` that copy was stored in the clear. `build2` fixed it,
by moving the scramble flag into each string's own header, and bumped
the bytecode format byte to say so.

In [ ]:
local chunk = load([[
  local ~function secure() return "SHARED-LITERAL" end
  local function ordinary() return "SHARED-LITERAL" end
  return secure, ordinary
]])

print("shared literal in the clear?",
  string.dump(chunk):find("SHARED-LITERAL", 1, true) ~= nil)

If that prints `true`, you are on a build older than `5.5.1_build2`.
Switch runtimes with the dropdown and run it again — that is the sort
of thing the runtime switcher is for.

## What it does not do

From Diluvium's own guide, quoted rather than paraphrased:

> Recovering the strings takes reading `ldump.c` and implementing the
> keystream, which is trivial for anyone who wants to. It raises the
> cost from one shell command to an afternoon. **Do not put a
> credential in one.**

This Lab has implemented that keystream — it is in
`src/analysis/luac.js`, because the bytecode viewer has to read every
dump it is given. So the demonstration is right here:

1. Run the cell below.
2. Press its **Bytecode** button.
3. Find the constants, and read the string that was "hidden".

Nothing runs when you press Bytecode — it compiles and parses only,
which is what makes it safe to point at a dump you were sent.

In [ ]:
local ~function looks_secure()
  local not_a_credential = "read me in the bytecode panel"
  return not_a_credential
end

print(looks_secure())

The scramble is a single-byte XOR up to bytecode format `0x45`, and a
length-seeded keystream from `0x46`. Both are reversible by design; the
second only stops one `tr` invocation recovering every hidden string in
a file at once.

**Use it for what it is for**: raising the cost of casually reading
strings out of a shipped binary. Not for secrets.

## Reading a dump you were sent

The **Bytecode** panel has a *Read hex* tab. Paste the hex of a
compiled chunk into it and the Lab will disassemble it — constants,
upvalues and jump targets resolved — without executing a byte.

Here is some hex to paste, so you can try it on something that did not
come from this page.

In [ ]:
local ~function example(n)
  local label = "computed"
  return label .. ": " .. (n * 2)
end

local dump = string.dump(example)
local hex = {}
for i = 1, #dump do hex[i] = string.format("%02x", dump:byte(i)) end
print(table.concat(hex))

Copy that output, press **Bytecode** on any cell, choose **Read hex**,
and paste it in.

The parser is strict on purpose: it refuses anything whose bytes it
cannot fully account for, rather than guessing and showing you a
plausible disassembly of something else.